### Chain using Langgraph
- Use chat Messages as Graph State
- Use as Chat Models in Graph Nodes
- Bind tools to LLM in chat models
- Execute the tolls in graph nodes

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


### How to use chat messages in Graph States

Messages are the fundamental communication unit in LangChain and LangGraph, used to represent interactions between the user, the model, and the system. 
The four major message types are:

- Human Message → User input
- AI Message → Model response
- System Message → Instructions for the model
- Tool Message → Tool execution results

Each message has 4 main attributes:
- content: The text content of the message.
- role: The role of the message sender (e.g., "user", "assistant", "system").
- additional_kwargs: A dictionary for any extra information related to the message.


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from pprint import pprint

messages = [AIMessage(content="Hello! How can I assist you today?", name="LLM")]
messages.append(HumanMessage(content="I want to learn about Langgraph." , name="John Doe"))
messages.append(AIMessage(content="Langgraph is a framework for building state graphs. It allows you to define nodes and edges, and then execute the graph based on the defined logic.", name="LLM"))
messages.append(HumanMessage(content="Can you give me an example of how to create a simple graph?", name="John Doe"))

for message in messages:
    print(f"{message.name}: {message.content}\n")

### Chat Models

we can use the sequence of messages as input with the chatmodels using LLM's and OPENAI


In [ ]:
from langchain_groq import ChatGroq

llm_groq = ChatGroq(model="qwen/qwen3-32b")
response = llm_groq.invoke(messages)
print(f"LLM: {response.content}")

# Tools

- Tools can be integrated with the LLM Models to interact with external systems . External systems can be APIs , third party tools.
- whenever a query is asked the model can choose to call the tool and this query is based on the natural language input and this will return an output that matches the tool's schema  

In [ ]:
def add(a:int , b:int) -> int:
    """
    Adds two integers and returns the result.
    Parameters:
    a (int): The first integer to add.
    b (int): The second integer to add.
    Returns:
    int: The sum of a and b.
    """
    return a+b

In [ ]:
# Binding tool with LLM

llm_with_tools = llm_groq.bind_tools([add])

tool_calls=llm_with_tools.invoke([HumanMessage(content="What is 2 + 2?")])

In [ ]:
tool_calls.tool_calls

### Using Messages as State

In [ ]:

from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

# Here we are using reducers add_message to ensure that any messages are appended to the existing lis of messages

from langgraph.graph.message import add_messages
from typing import Annotated

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    name:str

### Reducers with add_messages

In [ ]:
initial_messages = [AIMessage(content="Hello! How can I assist you today?", name="LLM")]
initial_messages.append(HumanMessage(content="I want to learn Python Programming." , name="John Doe"))
initial_messages

ai_message = AIMessage(content="Python is a versatile programming language that is widely used for web development, data analysis, artificial intelligence, and more.", name="LLM")

add_messages(initial_messages, ai_message) # This will append the ai_message to the initial_messages list instead of overwriting it.

In [ ]:
### chabot node function
def llm_node(state:State):
    return {"messages": llm_with_tools.invoke(state["messages"])}

In [ ]:
### State graph definition
from langgraph import graph
from langgraph.graph import END, START, StateGraph
from IPython.display import display, Image

builder=StateGraph(State)

# creating a node for the llm
builder.add_node("LLM_Node", llm_node)

# Add edges

builder.add_edge(START,"LLM_Node")
builder.add_edge("LLM_Node", END)

# Compile the graph
graph_builder = builder.compile()

# View the graph
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
# Invoke the graph
messages = graph_builder.invoke({"messages": " what is 3 + 5?"})
print(messages)

In [ ]:
# so far, LLM has decided a tool call is required . But, it has not executed the tool yet. This is because , there is no node in the graph which can execute the tool call. We can create a node which can execute the tool call and add it to the graph.
from langgraph.prebuilt import ToolNode, tools_condition

tools = [add]

builder = StateGraph(State)

builder.add_node("LLM_Node", llm_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "LLM_Node")

builder.add_conditional_edges( # This will add a conditional edge from LLM Node to the tools node based on whether the LLM has decided to make a tool call or not.
    "LLM_Node",
    tools_condition
)

builder.add_edge("tools", "LLM_Node")

builder.add_edge("LLM_Node", END)

graph_builder = builder.compile()

# View the graph
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
# Execute the graph
result = graph_builder.invoke({
    "messages": [HumanMessage(content="What is 3 + 5?")]
})

for msg in result["messages"]:
    print(type(msg).__name__)
    print(msg.content)
    print("-----")

In [ ]:
# Execute the graph
result = graph_builder.invoke({
    "messages": [HumanMessage(content="What is Machine Learning?")] # No tool call will be made here as the LLM will not decide to make a tool call for this query.
})

for msg in result["messages"]:
    print(type(msg).__name__)
    print(msg.content)
    print("-----")